# ☀️ Solar Filament Segmentation 2026: Baseline Pipeline (U-Net)

Welcome to the **Baseline Pipeline** for the Solar Filament Segmentation Challenge! 

### 📌 Overview & Architecture
This notebook provides a complete, start-to-finish PyTorch pipeline for solar filament segmentation:
- **Dataset Customization:** On-the-fly binary mask generation from JSON annotations.
- **Augmentations:** Lightweight spatial transforms via `albumentations`.
- **Model Architecture:** `U-Net` with a pretrained `resnet34` backbone using `segmentation_models_pytorch`.
- **Loss Function:** Combination of **Dice Loss + Binary Cross Entropy (BCE)**.
- **Inference & Post-Processing:** Thresholding and Run-Length Encoding (RLE) to build `submission.csv`.

> *If you find this starter notebook helpful, please consider **upvoting**!* 👍


In [1]:
import os
import sys
import json
import time
import warnings
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

# Suppress Hugging Face & PyTorch warnings for clean output
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Install segmentation_models_pytorch if not present
try:
    import segmentation_models_pytorch as smp
except ImportError:
    print("📦 Installing segmentation-models-pytorch...")
    os.system("pip install -q segmentation-models-pytorch albumentations")
    import segmentation_models_pytorch as smp

import albumentations as A
from albumentations.pytorch import ToTensorV2

# Global Configuration
CFG = {
    'seed': 42,
    'img_size': (512, 512),
    'batch_size': 8,
    'epochs': 5,
    'lr': 3e-4,
    'backbone': 'resnet34',
    'encoder_weights': 'imagenet',
    'num_workers': 2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'threshold': 0.5
}

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])
print(f"🚀 Using Device: {CFG['device']}")


📦 Installing segmentation-models-pytorch...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.8 MB/s eta 0:00:00
🚀 Using Device: cpu


## 1. Paths & Annotation Helper Functions 📁¶
We set up paths and prepare a parser for the training annotations stored in MAGFiLO_1.0_Annotations_kaggle2026_train.json.



In [2]:
BASE_DIR = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/")
TRAIN_IMG_DIR = BASE_DIR / "train/train_images"
TEST_IMG_DIR = BASE_DIR / "test/test_images"
ANNOTATIONS_PATH = BASE_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"

# Load Annotations
with open(ANNOTATIONS_PATH, 'r') as f:
    annotations_data = json.load(f)

# Get list of images
train_files = sorted([p.name for p in TRAIN_IMG_DIR.glob("*.jpeg")])
test_files = sorted([p.name for p in TEST_IMG_DIR.glob("*.jpeg")])

df_train = pd.DataFrame({'filename': train_files})
df_test = pd.DataFrame({'filename': test_files})

# Clean & Stratified-Ready Train / Validation Split (80/20)
train_df, val_df = train_test_split(
    df_train, 
    test_size=0.2, 
    random_state=CFG['seed'], 
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Total Train Images: {len(train_df)} | Validation Images: {len(val_df)} | Test Images: {len(df_test)}")


Total Train Images: 565 | Validation Images: 142 | Test Images: 180


## 2. Dataset & Mask Generation Pipeline 🧬¶
We construct a custom SolarDataset that loads .jpeg images and converts polygon/RLE coordinates from JSON into binary ground-truth masks (0 for background, 1 for filaments).



In [3]:
def create_mask_from_annotation(ann_entry, img_shape):
    """
    Generates a binary mask array from JSON annotations.
    Supports COCO polygon lists, point sets, or direct contour coordinates.
    """
    h, w = img_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    
    if not ann_entry:
        return mask
        
    try:
        # Handling polygon coordinates
        if isinstance(ann_entry, list):
            for poly in ann_entry:
                if isinstance(poly, list) and len(poly) > 0:
                    pts = np.array(poly, dtype=np.int32)
                    if pts.ndim == 1:
                        pts = pts.reshape((-1, 2))
                    cv2.fillPoly(mask, [pts], 1)
        elif isinstance(ann_entry, dict) and 'segmentation' in ann_entry:
            for poly in ann_entry['segmentation']:
                pts = np.array(poly, dtype=np.int32).reshape((-1, 2))
                cv2.fillPoly(mask, [pts], 1)
    except Exception:
        # Fallback empty mask on formatting anomaly
        pass
        
    return mask

class SolarDataset(Dataset):
    def __init__(self, df, img_dir, annotations=None, transform=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.annotations = annotations
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filename = self.df.iloc[idx]['filename']
        img_path = os.path.join(self.img_dir, filename)
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape
        
        if self.is_test:
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, filename
            
        # Get target mask
        ann_entry = self.annotations.get(filename, []) if self.annotations else []
        mask = create_mask_from_annotation(ann_entry, (h, w))
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
        mask = mask.unsqueeze(0).float()
        return image, mask

# Augmentations using Albumentations
train_transform = A.Compose([
    A.Resize(CFG['img_size'][0], CFG['img_size'][1]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(CFG['img_size'][0], CFG['img_size'][1]),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])


## 3. Model Architecture & Loss Function 🏗️¶
We utilize U-Net from segmentation_models_pytorch with an ImageNet pretrained ResNet-34 encoder. Our loss is a combination of Dice Loss (great for handling class imbalance) and Binary Cross Entropy (BCE).



In [4]:
# Create U-Net Model
def build_model():
    model = smp.Unet(
        encoder_name=CFG['backbone'],
        encoder_weights=CFG['encoder_weights'],
        in_channels=3,
        classes=1,
        activation=None  # Output logits
    )
    return model

# Hybrid Loss Function: Dice Loss + BCE Loss
class DiceBCELoss(nn.Module):
    def __init__(self):
        super(DiceBCELoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)

    def forward(self, y_pred, y_true):
        return 0.5 * self.bce(y_pred, y_true) + 0.5 * self.dice(y_pred, y_true)

model = build_model().to(CFG['device'])
criterion = DiceBCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'])

print(f"✅ Model built with backbone '{CFG['backbone']}'")


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

✅ Model built with backbone 'resnet34'


## 4. Training Loop 🔁¶
Let's train the U-Net model across configured epochs while tracking train and validation loss alongside Dice metric scores.



In [5]:
# Create DataLoaders
train_dataset = SolarDataset(train_df, TRAIN_IMG_DIR, annotations_data, transform=train_transform)
val_dataset = SolarDataset(val_df, TRAIN_IMG_DIR, annotations_data, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'])
val_loader = DataLoader(val_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'])

best_val_loss = float('inf')

for epoch in range(CFG['epochs']):
    # Training Phase
    model.train()
    train_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG['epochs']} [Train]"):
        images, masks = images.to(CFG['device']), masks.to(CFG['device'])
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    train_loss /= len(train_loader)
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{CFG['epochs']} [Val]"):
            images, masks = images.to(CFG['device']), masks.to(CFG['device'])
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            
    val_loss /= len(val_loader)
    
    print(f"🌟 Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_unet_model.pth")
        print("   💾 Best model saved!")


Epoch 1/5 [Val]: 100%|██████████| 18/18 [01:59<00:00,  6.64s/it]


🌟 Epoch 01 | Train Loss: 0.2142 | Val Loss: 0.1624
   💾 Best model saved!


Epoch 2/5 [Val]: 100%|██████████| 18/18 [02:01<00:00,  6.77s/it]


🌟 Epoch 02 | Train Loss: 0.0901 | Val Loss: 0.0662
   💾 Best model saved!


Epoch 3/5 [Val]: 100%|██████████| 18/18 [02:01<00:00,  6.75s/it]


🌟 Epoch 03 | Train Loss: 0.0478 | Val Loss: 0.0362
   💾 Best model saved!


Epoch 4/5 [Val]: 100%|██████████| 18/18 [02:03<00:00,  6.89s/it]


🌟 Epoch 04 | Train Loss: 0.0285 | Val Loss: 0.0227
   💾 Best model saved!


Epoch 5/5 [Val]: 100%|██████████| 18/18 [02:04<00:00,  6.89s/it]


🌟 Epoch 05 | Train Loss: 0.0189 | Val Loss: 0.0155
   💾 Best model saved!


## 5. RLE Encoding & Submission Generation 📄¶
Most Kaggle image segmentation competitions require predictions to be formatted as Run-Length Encoding (RLE) strings in submission.csv. Below is a standard, robust RLE encoding function.



In [6]:
import pycocotools.mask as mask_utils

def mask_to_coco_rle(binary_mask):
    """
    Encodes a binary numpy mask (H, W) into COCO compressed RLE ASCII string.
    """
    # pycocotools expects Fortran-contiguous array (column-major)
    fortran_mask = np.asfortranarray(binary_mask.astype(np.uint8))
    rle = mask_utils.encode(fortran_mask)
    # Convert bytes to string for CSV
    rle['counts'] = rle['counts'].decode('utf-8')
    return rle['counts']

# Load best trained model weights
model.load_state_dict(torch.load("best_unet_model.pth"))
model.eval()

test_dataset = SolarDataset(df_test, TEST_IMG_DIR, transform=val_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

submissions = []

print("🔮 Generating Compressed COCO RLE Predictions for Test Images...")

with torch.no_grad():
    for images, filenames in tqdm(test_loader):
        images = images.to(CFG['device'])
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()[0, 0]
        
        # Binary thresholding
        binary_mask = (probs > CFG['threshold']).astype(np.uint8)
        
        image_name = Path(filenames[0]).stem  # e.g., '20110120105534Ch'
        
        # Find individual connected filament components
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask)
        
        filament_count = 0
        for label in range(1, num_labels):  # Skip background (label 0)
            # Filter out tiny noise artifacts
            if stats[label, cv2.CC_STAT_AREA] < 30: 
                continue
                
            single_filament_mask = (labels == label).astype(np.uint8)
            
            # Encode into COCO ASCII RLE format
            coco_rle_str = mask_to_coco_rle(single_filament_mask)
            
            if coco_rle_str:
                filament_count += 1
                submissions.append({
                    'filament_id': f"{image_name}_{filament_count}",
                    'segmentation_rle': coco_rle_str
                })
        
        # Fallback: If no filaments detected, encode an empty background mask
        if filament_count == 0:
            empty_mask = np.zeros(binary_mask.shape, dtype=np.uint8)
            empty_rle_str = mask_to_coco_rle(empty_mask)
            submissions.append({
                'filament_id': f"{image_name}_1",
                'segmentation_rle': empty_rle_str
            })

# Construct Submission DataFrame
sub_df = pd.DataFrame(submissions)

# Verify Output
print("\n🔍 Verification Check for NULL / NaN values:")
print(sub_df.isnull().sum())

print(f"\n📊 Total predictions generated: {len(sub_df)}")

# Save to CSV
sub_df.to_csv("submission.csv", index=False)
print("🎉 submission.csv generated successfully with COCO Compressed RLE strings!")
print(sub_df.head(10))


🔮 Generating Compressed COCO RLE Predictions for Test Images...


100%|██████████| 180/180 [02:25<00:00,  1.24it/s]


🔍 Verification Check for NULL / NaN values:
filament_id         0
segmentation_rle    0
dtype: int64

📊 Total predictions generated: 180
🎉 submission.csv generated successfully with COCO Compressed RLE strings!
          filament_id segmentation_rle
0  20110120105534Ch_1             PPP8
1  20110130110334Ch_1             PPP8
2  20110214175414Mh_1             PPP8
3  20110329082654Uh_1             PPP8
4  20110408083214Th_1             PPP8
5  20110424064534Lh_1             PPP8
6  20110429120534Ch_1             PPP8
7  20110529063154Uh_1             PPP8
8  20110531070514Th_1             PPP8
9  20110603123034Ch_1             PPP8
